# Cross-Crisis Phase Comparison

This notebook compares crises on an aligned timeline by phase start (`month_from_phase_start = 0`).

It lets you answer questions like:
- How quickly did labor damage appear after the **system break** across crises?
- Which crisis had the sharpest credit stress escalation during **cracks**?


In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

ROOT = Path('..')

panel = pd.read_csv(ROOT / 'data/processed/indicator_panel_monthly.csv', parse_dates=['date'])
catalog = pd.read_csv(ROOT / 'data/reference/fred_series_catalog.csv')
phase_windows = pd.read_csv(ROOT / 'data/reference/crisis_phase_windows.csv', parse_dates=['start_date', 'end_date'])
crises = pd.read_csv(ROOT / 'data/reference/crisis_catalog.csv')

panel.head()

## Available Crises and Phase Windows

In [ ]:
display(crises[['crisis_id', 'crisis_name', 'region']])

display(
    phase_windows.sort_values(['crisis_id', 'phase_order'])
    [['crisis_id', 'phase', 'start_date', 'end_date']]
)

## Configuration

In [ ]:
# Choose comparison slice
TARGET_PHASE = 'system_break'  # pre_crisis, cracks, system_break, spillover
USE_ZSCORE = True

# Option A: explicit indicator list
SELECTED_SERIES = ['TEDRATE', 'VIXCLS', 'NASDAQCOM', 'BAA10Y', 'BAMLH0A0HYM2']

# Option B: leave SELECTED_SERIES empty and use domains instead
SELECTED_DOMAINS = []  # e.g. ['credit', 'markets', 'liquidity']

# Optional clipping to first N months from phase start
MAX_MONTH_FROM_START = None  # e.g. 18

In [ ]:
def month_index(date_series: pd.Series, start_date: pd.Timestamp) -> pd.Series:
    return (date_series.dt.year - start_date.year) * 12 + (date_series.dt.month - start_date.month)


def build_aligned_phase_panel(
    panel_df: pd.DataFrame,
    catalog_df: pd.DataFrame,
    phase_windows_df: pd.DataFrame,
    phase: str,
    use_zscore: bool = True,
    selected_series=None,
    selected_domains=None,
    max_month_from_start=None,
) -> pd.DataFrame:
    selected_series = selected_series or []
    selected_domains = selected_domains or []

    value_col = 'value_z' if use_zscore else 'value'
    phase_rows = phase_windows_df.loc[phase_windows_df['phase'] == phase].copy()
    if phase_rows.empty:
        raise ValueError(f'No phase rows found for phase={phase}')

    frames = []
    for _, row in phase_rows.iterrows():
        cid = row['crisis_id']
        start = row['start_date']
        end = row['end_date']

        block = panel_df.loc[(panel_df['date'] >= start) & (panel_df['date'] <= end)].copy()
        block['crisis_id'] = cid
        block['phase'] = phase
        block['phase_start'] = start
        block['phase_end'] = end
        block['month_from_phase_start'] = month_index(block['date'], start)
        frames.append(block)

    aligned = pd.concat(frames, ignore_index=True)

    if selected_series:
        aligned = aligned.loc[aligned['series_id'].isin(selected_series)].copy()
    elif selected_domains:
        aligned = aligned.loc[aligned['domain'].isin(selected_domains)].copy()

    if max_month_from_start is not None:
        aligned = aligned.loc[aligned['month_from_phase_start'] <= max_month_from_start].copy()

    aligned = aligned.dropna(subset=[value_col]).copy()
    aligned['metric'] = aligned[value_col]

    # Attach readable names
    aligned = aligned.merge(
        catalog_df[['series_id', 'indicator_name', 'domain', 'phase_default']],
        on=['series_id', 'domain', 'phase_default'],
        how='left',
    )

    aligned = aligned.merge(
        crises[['crisis_id', 'crisis_name', 'region']],
        on='crisis_id',
        how='left',
    )

    return aligned.sort_values(['series_id', 'crisis_id', 'month_from_phase_start'])

In [ ]:
aligned = build_aligned_phase_panel(
    panel_df=panel,
    catalog_df=catalog,
    phase_windows_df=phase_windows,
    phase=TARGET_PHASE,
    use_zscore=USE_ZSCORE,
    selected_series=SELECTED_SERIES,
    selected_domains=SELECTED_DOMAINS,
    max_month_from_start=MAX_MONTH_FROM_START,
)

print(f'Rows: {len(aligned):,}')
print('Series:', sorted(aligned['series_id'].unique()))
print('Crises:', sorted(aligned['crisis_id'].unique()))
aligned.head()

## Aligned Series Comparison (Phase Start = Month 0)

In [ ]:
y_label = 'z-score' if USE_ZSCORE else 'raw value'

fig = px.line(
    aligned,
    x='month_from_phase_start',
    y='metric',
    color='crisis_name',
    facet_row='series_id',
    line_dash='crisis_name',
    height=max(550, 230 * aligned['series_id'].nunique()),
    markers=True,
    title=f'Cross-crisis comparison: {TARGET_PHASE} phase ({y_label})',
)
fig.update_yaxes(matches=None)
fig.update_xaxes(title='Months since phase start')
fig.for_each_yaxis(lambda axis: axis.update(title=y_label))
fig.update_layout(legend_title='Crisis', margin=dict(l=40, r=40, t=80, b=30))
fig.show()

## Domain-Level Composite (Average z-score)

In [ ]:
if not USE_ZSCORE:
    raise ValueError('Domain-level composite is best interpreted with USE_ZSCORE=True')

domain_comp = (
    aligned.groupby(['crisis_name', 'domain', 'month_from_phase_start'], as_index=False)['metric']
    .mean()
)

fig = px.line(
    domain_comp,
    x='month_from_phase_start',
    y='metric',
    color='crisis_name',
    facet_row='domain',
    line_dash='crisis_name',
    markers=True,
    height=max(500, 220 * domain_comp['domain'].nunique()),
    title=f'Domain composite comparison: {TARGET_PHASE} phase',
)
fig.update_yaxes(matches=None)
fig.update_xaxes(title='Months since phase start')
fig.for_each_yaxis(lambda axis: axis.update(title='mean z-score'))
fig.update_layout(legend_title='Crisis', margin=dict(l=40, r=40, t=80, b=30))
fig.show()

## Tips

- For a clean apples-to-apples view, compare one domain at a time.
- Use `MAX_MONTH_FROM_START` (e.g., 12) to compare first-year dynamics.
- Keep `USE_ZSCORE=True` for cross-indicator comparability.
